# Inez CO₂ Storage Capacity Assessment

This notebook runs a probabilistic static storage-capacity assessment for the **Haldager Sand, Gassum and Skagerrak** reservoirs, then adds their simulated capacities trial by trial to obtain the combined Inez distribution.

$$SC = GRV \times (N/G) \times \phi \times \rho_{CO_2} \times S_{eff}$$

Change the values in the **Editable inputs** cell, then choose **Runtime → Run all**. The code used for calculations and figures is collapsed by default; its tables, plots and results remain visible.


In [ ]:
#@title Install dependencies { display-mode: "form" }
"""Install the latest package and plotting tools from GitHub."""
%pip install -q --upgrade --force-reinstall --no-cache-dir --no-deps "git+https://github.com/AnaSoles/ggg-co2-storage-eval.git"
%pip install -q matplotlib pandas


In [ ]:
#@title Load analysis tools { display-mode: "form" }
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from storageeval import Distribution, SimulationResult, StorageSite, simulate

plt.style.use("seaborn-v0_8-whitegrid")


## Editable inputs

Enter minimum, most likely and maximum values. Fractions are decimals: `0.07` means 7%. The values come from GEUS Report 2022/29, Tables 8.1.5.1–8.1.5.3.

**Source note:** the Inez Gassum reservoir-specific table gives a 7% storage-efficiency mode. This value reproduces the detailed published result more closely than the 10% mentioned in the report's general text.


In [ ]:
#@title Editable inputs — keep this cell visible
inputs = {
    "Haldager Sand": {
        "GRV (km³)": (0.114, 0.417, 1.676), "N/G": (0.200, 0.317, 0.500),
        "Porosity": (0.200, 0.255, 0.300), "CO₂ density (kg/m³)": (609.6, 641.7, 705.9),
        "Storage efficiency": (0.050, 0.100, 0.150)},
    "Gassum Formation": {
        "GRV (km³)": (11.059, 28.076, 49.666), "N/G": (0.4696, 0.5870, 0.7044),
        "Porosity": (0.1624, 0.2030, 0.2436), "CO₂ density (kg/m³)": (611.8, 644.0, 708.4),
        "Storage efficiency": (0.050, 0.070, 0.150)},
    "Skagerrak Formation": {
        "GRV (km³)": (3.7105, 8.781, 12.915), "N/G": (0.3056, 0.3820, 0.4584),
        "Porosity": (0.1624, 0.2030, 0.2436), "CO₂ density (kg/m³)": (607.2, 639.2, 703.1),
        "Storage efficiency": (0.050, 0.100, 0.150)},
}


## Input parameter table

This table is generated from the editable values above, so it updates automatically when an input changes.


In [ ]:
#@title Show input parameter table { display-mode: "form" }
rows = [[reservoir, parameter, "PERT", *values] for reservoir, parameters in inputs.items() for parameter, values in parameters.items()]
input_table = pd.DataFrame(rows, columns=["Reservoir", "Parameter", "Distribution", "Minimum", "Mode", "Maximum"])
input_table


In [ ]:
#@title Run the Monte Carlo simulation { display-mode: "form" }
def make_site(name, values):
    return StorageSite(
        name=f"Inez – {name}",
        grv=Distribution.pert(*values["GRV (km³)"]),
        net_to_gross=Distribution.pert(*values["N/G"]),
        porosity=Distribution.pert(*values["Porosity"]),
        co2_density=Distribution.pert(*values["CO₂ density (kg/m³)"]),
        storage_efficiency=Distribution.pert(*values["Storage efficiency"]),
    )

iterations = 100_000
results = {name: simulate(make_site(name, values), iterations, seed=42+i) for i, (name, values) in enumerate(inputs.items())}
combined_capacity = np.sum([result.capacity_mt for result in results.values()], axis=0)
combined = SimulationResult("Inez – combined reservoirs", combined_capacity, {})


## Results and comparison with GEUS

P90/P50/P10 are calculated from each final capacity distribution. For combined Inez, the three reservoirs are added in every trial and percentiles are calculated afterwards; reservoir percentiles are not added together.


In [ ]:
#@title Show capacity results { display-mode: "form" }
published = {
    "Haldager Sand": (1.2, 2.8, 5.5, 3.1),
    "Gassum Formation": (103.8, 168.1, 263.7, 177.6),
    "Skagerrak Formation": (27.4, 42.1, 60.7, 43.2),
    "Combined Inez": (148.6, 216.2, 310.2, 224.8),
}
all_results = {**results, "Combined Inez": combined}
comparison_rows = []
for name, result in all_results.items():
    s = result.summary()
    simulated = (s["p90_mt"], s["p50_mt"], s["p10_mt"], s["mean_mt"])
    comparison_rows.append([name, *simulated, *published[name]])
comparison_table = pd.DataFrame(comparison_rows, columns=["Reservoir", "P90 simulated", "P50 simulated", "P10 simulated", "Mean simulated", "P90 GEUS", "P50 GEUS", "P10 GEUS", "Mean GEUS"])
comparison_table.round(2)


The notebook and GEUS values should be close but not necessarily identical. Both use the same static volumetric equation and independent PERT inputs. Small differences are expected because the report does not state its random seed, iteration count or exact PERT implementation.


## Input uncertainty distributions


In [ ]:
#@title Show input uncertainty distributions { display-mode: "form" }
labels = {
    "grv_km3": "GRV (km³)",
    "net_to_gross": "Net-to-gross",
    "porosity": "Porosity",
    "co2_density_kg_m3": "CO₂ density (kg/m³)",
    "storage_efficiency": "Storage efficiency",
}
fig, axes = plt.subplots(5, 3, figsize=(15, 16))
for column, (reservoir_name, result) in enumerate(results.items()):
    for row, (name, values) in enumerate(result.inputs.items()):
        ax = axes[row, column]
        ax.hist(values, bins=45, color="#2a6fbb", alpha=0.82)
        ax.set_title(f"{reservoir_name}\n{labels[name]}")
        ax.set_ylabel("Simulations")
fig.suptitle("Inez – input uncertainty distributions", fontsize=16, y=1.01)
fig.tight_layout()
plt.show()


## Combined storage-capacity distribution and cumulative curve

The grey bars show simulated capacities, the orange line is a fitted lognormal probability-density curve, and the red line is the probability that capacity is exceeded. P90 is the conservative estimate, P50 the median and P10 the upside estimate.


In [ ]:
#@title Show combined histogram and cumulative curve { display-mode: "form" }
values = np.asarray(combined.capacity_mt, dtype=float)
summary = combined.summary()
fig, ax_density = plt.subplots(figsize=(12, 7))
counts, bins, patches = ax_density.hist(values, bins=55, density=True, color="#d9d9d9", edgecolor="white", linewidth=0.55, label="Simulated capacity")

log_values = np.log(values)
mu = float(np.mean(log_values))
sigma = float(np.std(log_values, ddof=1))
x = np.linspace(float(np.min(values)), float(np.max(values)), 600)
fitted_density = np.exp(-0.5 * ((np.log(x) - mu) / sigma) ** 2) / (x * sigma * np.sqrt(2 * np.pi))
ax_density.plot(x, fitted_density, color="#f39c12", linewidth=2.2, label="Fitted lognormal density")

ax_cumulative = ax_density.twinx()
capacity_sorted = np.sort(values)
exceedance_pct = (1 - np.arange(1, capacity_sorted.size + 1) / (capacity_sorted.size + 1)) * 100
ax_cumulative.plot(capacity_sorted, exceedance_pct, color="#e31a1c", linewidth=2.0, label="Cumulative exceedance")

markers = [("P90", "p90_mt", 90, "#e31a1c"), ("P50", "p50_mt", 50, "#ff8c42"), ("P10", "p10_mt", 10, "#e31a1c")]
for label, key, probability, color in markers:
    value = summary[key]
    ax_density.axvline(value, color=color, linestyle="--", linewidth=1.25, alpha=0.9)
    ax_cumulative.plot(value, probability, "o", color=color, markersize=5)
    ax_cumulative.annotate(f"{label}: {value:.1f} Mt", (value, probability), xytext=(6, 7), textcoords="offset points", color=color, fontsize=9)

ax_density.text(0.99, 0.97, f"Mean = {summary['mean_mt']:.1f} Mt; SD = {np.std(values, ddof=1):.1f} Mt", transform=ax_density.transAxes, ha="right", va="top", color="#d97904")
ax_density.set(xlabel="Storage capacity (Mt CO₂)", ylabel="Probability density", title="Inez – combined storage-capacity distribution")
ax_cumulative.set(ylabel="Cumulative exceedance probability (%)", ylim=(0, 100))
ax_cumulative.tick_params(axis="y", colors="#e31a1c")
ax_cumulative.yaxis.label.set_color("#e31a1c")
handles_1, labels_1 = ax_density.get_legend_handles_labels()
handles_2, labels_2 = ax_cumulative.get_legend_handles_labels()
ax_density.legend(handles_1 + handles_2, labels_1 + labels_2, loc="upper right", bbox_to_anchor=(1, 0.89))
fig.tight_layout()
plt.show()


## Exceedance curve

P90 is the capacity that has a 90% probability of being exceeded; P10 is the upside estimate.


In [ ]:
#@title Show exceedance curve { display-mode: "form" }
fig, ax = combined.plot_exceedance()
plt.show()


## Linear capacity confidence ranges

The colored bar summarizes conservative, central and upside capacity ranges. These are probabilistic static capacity estimates, not booked reserves.


In [ ]:
#@title Show linear capacity confidence ranges { display-mode: "form" }
fig, ax = combined.plot_capacity_ranges()
plt.show()


## Combined sensitivity tornado chart

The chart ranks all 15 uncertain inputs by their Spearman rank correlation with combined Inez capacity. Longer bars indicate a stronger association with capacity variation. The value printed on each bar is a correlation coefficient, not a percentage contribution.


In [ ]:
#@title Show combined sensitivity tornado chart { display-mode: "form" }
combined_sensitivity_inputs = {
    f"{reservoir_name} — {labels[input_name]}": samples
    for reservoir_name, result in results.items()
    for input_name, samples in result.inputs.items()
}
output_rank = np.argsort(np.argsort(combined.capacity_mt))
correlations = {}
for name, samples in combined_sensitivity_inputs.items():
    input_rank = np.argsort(np.argsort(samples))
    correlations[name] = float(np.corrcoef(input_rank, output_rank)[0, 1])

ordered = sorted(correlations.items(), key=lambda item: abs(item[1]))
names, sensitivity_values = zip(*ordered)
fig, ax = plt.subplots(figsize=(11, 8))
bars = ax.barh(names, sensitivity_values, color="#2a6fbb")
ax.axvline(0, color="black", linewidth=0.8)
for bar, value in zip(bars, sensitivity_values):
    inside = abs(value) >= 0.08
    ax.text(value - 0.012 if inside else value + 0.012, bar.get_y() + bar.get_height() / 2, f"{value:.3f}", ha="right" if inside else "left", va="center", color="white" if inside else "#1f1f1f", fontsize=8, fontweight="bold")
ax.set(xlabel="Spearman rank correlation", title="Inez – combined input sensitivity", xlim=(-1, 1))
fig.tight_layout()
plt.show()


## Source and limitation

Source: [GEUS Report 2022/29](https://data.geus.dk/pure-pdf/GEUS-R_2022-29_web.pdf), input Tables 8.1.5.1–8.1.5.3 (report page 44) and results Tables 8.2.1–8.2.4 (page 46).

This is static volumetric screening capacity. It does not yet represent pressure constraints, injectivity, plume migration, dynamic reservoir simulation or economics.
